# Addition Feature Engineering

This notebook conducts some additional feature engineering

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
import os
from pprint import pprint
import pickle
import math
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.manifold import TSNE

from util_IO import (
    load_pickle_from_main_project_dir,
    load_attributes_df,
    load_timeseries_df
)

# Set pandas to display a maximum of 300 columns
pd.set_option('display.max_columns', 300)
pd.set_option('display.max_rows', 1000)

# Suppress the SettingWithCopyWarning
pd.options.mode.chained_assignment = None

## Parameters

### Load metadata from

In [ ]:
aggr_parameters_dict, camels_gb_use_case_dir = load_pickle_from_main_project_dir(
    'aggr_parameters_dict.pkl'
)

### Retrieve variables in use

In [ ]:
camels_gb_data_attributes_aggr_dir = aggr_parameters_dict['camels_gb_data_attributes_aggr_dir']
attributes_index = aggr_parameters_dict["attributes"]["attributes_index"]
camels_gb_data_timeseries_aggr_dir = aggr_parameters_dict['camels_gb_data_timeseries_aggr_dir']
date_field = aggr_parameters_dict["timeseries"]["date_field"]
label_field = aggr_parameters_dict["timeseries"]['label_field']

# Attributes

## Retrieve aggregated file

In [ ]:
# Attributes
attributes_df = load_attributes_df(
    camels_gb_data_attributes_aggr_dir,
    "fundamental_postEDA.csv",
    attributes_index
)

display(attributes_df.head(3))

## Detach locations and names

In [ ]:
# Define interested fields
locations_names_fields_list = [
    "gauge_name",
    "gauge_lat",
    "gauge_lon"
]

# Defining a separated data frame for this kind of information
attributes_location_names_df = attributes_df[locations_names_fields_list]

# Remove the field from the data frame which feeds the machine model
attributes_df.drop(
    columns=locations_names_fields_list,
    inplace=True
)

In [ ]:
attributes_location_names_df.head()

### Locations and names data frame elaboration

In [ ]:
# Assert that each element in 'gauge_name' column contains exactly one 'at'
assert attributes_location_names_df['gauge_name'].apply(lambda x: x.count(' at ') == 1).all(), "Not all elements contain exactly one ' at '"

# Split 'gauge_name' column into two new columns 'tributary' and 'main'
location_split = (
    attributes_location_names_df['gauge_name']
        .str
        .split(' at ', expand=True)
        .rename(
            columns={0: "river", 1:"location"}
        )
)

# Concatenate
attributes_location_names_df = pd.concat(
    [attributes_location_names_df, location_split],
    axis=1
)


# Remove composed name
attributes_location_names_df.drop(
    columns='gauge_name',
    inplace=True
)

display(attributes_location_names_df.head(3))

#### Count number of location per river

In [ ]:
rivers_count_df = (
    attributes_location_names_df
        .groupby('river')
        .size()
        .reset_index(name='count')
        .sort_values(by='count', ascending=False)
)

# Change to `True` if you want to display counts on rivers
if False:   
    display(rivers_count_df)

# Timeseries

## Retrieve aggregated file

In [ ]:
timeseries_df = load_timeseries_df(
    camels_gb_data_timeseries_aggr_dir,
    "timeseries_postEDA.csv",
    date_field
)

display(timeseries_df.head(3))

## Add label transformation(s)

In [ ]:
# log1p transformation
timeseries_df[f"log1p_{label_field}"] = np.log1p(timeseries_df[label_field])

## Define "beginning of the year"

In [ ]:
# Set variables
year_start_month = 3
year_start_day = 21

## Add sin() and cos() functions to map the moment throughout the year

In [ ]:
# ___________________________________
# Convert date column in Unix seconds
unix_sec_series = (
    timeseries_df[date_field]
        .apply(
            lambda x:
                int(
                    datetime.datetime.combine(x, datetime.datetime.min.time())
                        .timestamp()
                )
        )
)

# ______________________________________
# Find the oldest "beginning of the year"
first_beginning_year = (
    timeseries_df
        .loc[
            (pd.to_datetime(timeseries_df[date_field]).dt.month == year_start_month) & 
                (pd.to_datetime(timeseries_df[date_field]).dt.day == year_start_day)
        ]
    [date_field]
    .min()
)

first_beginning_year_unix_sec = pd.Timestamp(first_beginning_year).timestamp()

# ______________________________________________________________________
# Centred Unix second series to 0 for the `first_beginning_year_unix_sec`
unix_sec_centred_series = unix_sec_series - first_beginning_year_unix_sec


# Set the n° of "days" in a year
sec_in_a_year = 60*60*24*(365.2425)

# Add sin() and cos()
timeseries_df['sin_year'] = np.sin(unix_sec_centred_series * (2 * np.pi / sec_in_a_year))
timeseries_df['cos_year'] = np.cos(unix_sec_centred_series * (2 * np.pi / sec_in_a_year))

### Visual check

In [ ]:
# Identify the catchmentID-group with the fewest rows but considerable amount of observations
combined_counts = timeseries_df.groupby(['catchmentID', f"{date_field}_group"]).size()

# Reducing to time series with at least (almost) 2 years
combined_counts = combined_counts[combined_counts > 600]

# Retrieve the shortest (for display matter)
min_index = combined_counts.idxmin()

# Filter the DataFrame
min_timeseries_df = (
    timeseries_df[
        (timeseries_df['catchmentID'] == min_index[0]) &
            (timeseries_df[f"{date_field}_group"] == min_index[1])
    ]
)

# Plot
plt.figure(figsize=(20, 5))
sns.lineplot(data=min_timeseries_df, x=date_field, y='sin_year', marker='o', label='sin(y)')
sns.lineplot(data=min_timeseries_df, x=date_field, y='cos_year', marker='o', label='cos(y)')
plt.title(f'Catchment ID: {min_index[0]}')
plt.xlabel('')
plt.ylabel('')
plt.legend()
plt.show()

## Add time reference

Add time reference, starting with 1 for the oldest observation, and increase by 1 for each following day.

In [ ]:
# Find the oldest date
oldest_date = pd.to_datetime(timeseries_df[date_field].min())
newest_date = pd.to_datetime(timeseries_df[date_field].max())

print(f"Oldest date within the dataset, regardless catchment ID:\t{oldest_date}")
print(f"Newest date within the dataset, regardless catchment ID:\t{newest_date}")

# Calculate the difference in days from the oldest date
timeseries_df['time_ref'] = (pd.to_datetime(timeseries_df[date_field]) - oldest_date).dt.days

### Sample check

In [ ]:
display(
    timeseries_df[[date_field, 'time_ref']]
        .sample(n=10)
        .sort_values(date_field)
)

### Add a scaled `time_ref` (***deprecated***)

In [ ]:
timeseries_df['time_ref_scaled'] = timeseries_df['time_ref'] / ((newest_date-oldest_date).days)

assert (timeseries_df['time_ref_scaled'].max()==1) and (timeseries_df['time_ref_scaled'].min()==0), "Something wrong during calculation of the dates and/or while scaling"

# Save

In [ ]:
# _____________
# attributes_df

# Define path to save
path = os.path.join(
        camels_gb_data_attributes_aggr_dir,
        "fundamental_postFEa.csv"
)

# Save attributes
attributes_df.to_csv(path)


# Define path to save
path = os.path.join(
        camels_gb_data_attributes_aggr_dir,
        "fundamental_locations_postFEa.csv"
)

# Save attributes locations
attributes_location_names_df.to_csv(path)

# _____________
# timeseries_df

# Define path to save
path = os.path.join(
        camels_gb_data_timeseries_aggr_dir,
        "timeseries_postFEa.csv"
)

# Save
timeseries_df.to_csv(
    path,
    index=False
)

# Custom analysis with the `whole_set_df`

## Whole variables set data frame construction

In [ ]:
whole_set_df = (
    timeseries_df
        .merge(
            attributes_df,
            left_on='catchmentID',
            right_index=True,
            how='left'
        )
)

assert whole_set_df.isna().sum().sum() == 0, "Some NaN in the sample, please check!"

whole_set_df.drop(
    columns=[
        'discharge_vol',
        'date_group',
        'log1p_discharge_vol',
        'time_ref_scaled',
        
    ],
    inplace=True
)

In [ ]:
display(whole_set_df.head())

## Cluster analysis

### Define data set for clustering

In [ ]:
cluster_set_df = (
    whole_set_df
        .drop(
            columns=[
                'catchmentID',
                'date',
                'precipitation',
                'temperature',
                'humidity',
                'shortwave_rad',
                'longwave_rad',
                'windspeed',
                'sin_year',
                'cos_year',
                'time_ref'
            ]
        )
        .drop_duplicates()
        .reset_index(drop=True)
)

hue_variables = cluster_set_df.columns.to_list()


display(cluster_set_df.head())

### Clustering

In [ ]:
# Set `n_components`
n_components = 2

# Initialize t-SNE
tsne = TSNE(
    n_components=2,
    learning_rate=200,
    perplexity=50,
    early_exaggeration=6.0,
    init='pca',
    max_iter=3000,
    random_state=82
)

# Retrieve data
X = (
    cluster_set_df
        .drop(
            columns=[
                'elev_mean',
                'elev_min',
                'elev_10',
                'elev_50',
                'elev_90',
                'elev_max'	
            ]
        )
        .values
)

# Fit and transform the data
X_embedded = tsne.fit_transform(X)

# Aggregate results to `whole_set_df`
for i in range(n_components):
    cluster_set_df[f"Component {i+1}"] = X_embedded[:, i]
    
display(cluster_set_df.head())

### Plot clustering with `hue`

In [ ]:
# Number of subplots
num_subplots = len(hue_variables)
num_columns = 2

# Calculate the number of rows needed
num_rows = (num_subplots + num_columns - 1) // num_columns

# Create a figure with subplots
fig, axes = plt.subplots(num_rows, num_columns, figsize=(5 * num_columns, 5 * num_rows))

# Flatten the axes array for easy iteration
axes = axes.flatten()

# Loop through the variables and create a subplot for each
for i, var in enumerate(hue_variables):
    sns.scatterplot(
        data=cluster_set_df,
        x="Component 1",
        y="Component 2",
        hue=var,
        palette='viridis',
        ax=axes[i]
    )
    axes[i].set_title(f"t-SNE - {var}")
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

    # Remove the ticks
    axes[i].tick_params(left=False, bottom=False)
    axes[i].set_xticklabels([])
    axes[i].set_yticklabels([])

    # Remove all spines
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)
    axes[i].spines['left'].set_visible(False)
    axes[i].spines['bottom'].set_visible(False)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Adjust layout
plt.tight_layout()

# Save the figure as a PNG file
plt.savefig(
    os.path.join(
    camels_gb_use_case_dir,
    "resources",
    "charts",
    "03-FEa",
    'tsne_scatter_plots.png'
    )   
)

# Show the plot
plt.show()

## Custom split

### Custom split definition

In [ ]:
focus_catchments_list = [
    # Insert here your list of catchments to be analyzed
]


whole_set_df["On focus"] = (
    whole_set_df['catchmentID']
        .isin(focus_catchments_list)
)

### Custom split visualization

#### Custom split visualization function

In [ ]:
def split_plot_distributions(
    df,
    boolean_column,
    max_columns_per_plot=5,
    save_path=None
):
    # Filter out string and date columns
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Calculate the number of plots needed
    num_plots = math.ceil(len(numeric_columns) / max_columns_per_plot)
    
    # Create subplots for each chunk of columns
    for i in range(num_plots):
        start_idx = i * max_columns_per_plot
        end_idx = (i + 1) * max_columns_per_plot
        columns_chunk = numeric_columns[start_idx:end_idx]
        
        # Create a figure with subplots
        fig, axes = plt.subplots(len(columns_chunk), 1, figsize=(10, len(columns_chunk) * 4))
        
        # Plot each column's KDE split by the boolean column
        for j, column in enumerate(columns_chunk):
            sns.kdeplot(
                data=df,
                x=column,
                hue=boolean_column,
                ax=axes[j],
                bw_adjust=1.5,
                fill=True,
                alpha=0.7,
                common_norm=False
            )
            axes[j].set_title(f'{column}')
            axes[j].set_xlabel('')
        
        # Adjust layout and show the plot
        plt.tight_layout()
        
        # Save the plot if save_path is provided
        if save_path:
            plt.savefig(f"{save_path}_part_{i + 1}.png", dpi=300)

        plt.show()

In [ ]:
# Define dir to save
path = os.path.join(
    camels_gb_use_case_dir,
    "resources",
    "charts",
    "03-FEa",
    "On_focus"
)

split_plot_distributions(
    whole_set_df,
    "On focus",
    max_columns_per_plot=7,
    save_path=path
)